# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 6: LLM Agents with LangChain</font>

# <font color="#003660">LLM Agents</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how LLM agents are build with LangChain and LangGraph.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [ ]:
!pip install -U langchain langchain-community langchain-openai langgraph

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [ ]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull qwen3:8b # takes around a minute

# Agent

We want to build an Agent that consists a router (graph).

* The chat model will decide to make a tool call or not based upon the user input
* A conditional edge routes to a node that calls our tool or simply ends the route with an end node
* If multiple tool calls are necessary as in a mathematical environment where we want to apply multiple additions, subtractions, multiplications or divisions, the model needs a route to jump back and call the tools again.

We can solve this by simply pass that `ToolMessage` *back to the model*?

We can let it either (1) call another tool or (2) respond directly.

This is the intuition behind [*Reason+Act (ReAct)*](https://doi.org/10.48550/arXiv.2210.03629), a general agent architecture.
  
* `reason` - let the model reason about the input (e.g., call a tool or just respond directly)
* `act` - let the model call specific tools
* `observe` - pass the tool output back to the model

* (`answer` - answer the question after iterations of reason-act-observe)


This [general purpose architecture](https://blog.langchain.dev/planning-for-agents/) can be applied to many types of tools.

![Screenshot 2024-08-21 at 12.45.43 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0b4a2c1e5e02f3e78b_agent2.png)

[Source: LangChain Academy GitHub](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

In [ ]:
LLM_NAME = "qwen3:8b"
BASE_URL = "http://127.0.0.1:11434/v1" # localhost
OPENAI_API_KEY = "ollama" #fake api key

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

@tool
def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

@tool
def subtract(a: int, b: int) -> int:
    """Subtract b from a.

    Args:
        a: first int
        b: second int
    """
    return a - b

tools = [add, subtract, multiply, divide]


llm = ChatOpenAI(
    model=LLM_NAME,
    base_url=BASE_URL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    seed=42
)

llm_with_tools = llm.bind_tools(tools)

Let's create our LLM and prompt it with the overall desired agent behavior.

In [ ]:
# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

We will use a `MessagesState` and define a `Tools` node with our list of tools.

The `Assistant` node is just our model with bound tools.

In [ ]:
# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}



This is how we develop a graph with LangGraph:

1. We create a graph builder.
2. Then we add a start node. This node takes the first input of a user.
3. We add an assistant node

In [ ]:
# Graph
builder = StateGraph(MessagesState)

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")

# Define nodes: these do the work
builder.add_node("assistant", assistant)

react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

The above image shows how a common LLM workflow is. Input (start node), model processing (assistant node), output (end node). This is not very amazing, isn't it? Let's run it.

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 10")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

As we can see above its a tool calling agent. Because we added tools before, the LLM calls tools but they are not actually called. Let's add a tools node.

In [ ]:
# this is what we already know
# Graph
builder = StateGraph(MessagesState)
# Define nodes: these do the work
builder.add_node("assistant", assistant)
# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")

##### this is new #####
# add a tool node with our tools
builder.add_node("tools", ToolNode(tools))
# add an edge between assistant and tools
builder.add_edge("assistant", "tools")

react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

Looks like this ends with a tool call. Makes not much sense, or does it? Let's try it out.

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 10")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

What happens in the tool node is simply to call a tool. Nice, now we need to get the tool response back into the model.

In [ ]:
# this is what we already know
# Graph
builder = StateGraph(MessagesState)
# Define nodes: these do the work
builder.add_node("assistant", assistant)
# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
# add a tool node with our tools
builder.add_node("tools", ToolNode(tools))
# add an edge between assistant and tools
builder.add_edge("assistant", "tools")

##### this is new #####
# add an edge between tools and assistant to get backwards
builder.add_edge("tools", "assistant")
# add an edge between assistant and END node to finally end this
builder.add_edge("assistant", END)

react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

Looks weird. Let's run this.

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 10")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

Wow, a recursion error. Who could have foreseen this with the loop in the graph above.

To prevent the graph from repeating tool call and tool response, the assistant node, which is simply the LLM with tools, needs what LangGraph calls a conditional edge.

This does simply:

**If** the latest message (result) from assistant is a tool call:
    
    -> tools_condition routes to tools

**Else** (if the latest message (result) from assistant is a not a tool call):

    -> tools_condition routes to END
    

In [ ]:
# this is what we already know
# Graph
builder = StateGraph(MessagesState)
# Define nodes: these do the work
builder.add_node("assistant", assistant)
# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
# add a tool node with our tools
builder.add_node("tools", ToolNode(tools))

##### this is new #####
builder.add_conditional_edges(
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    tools_condition,
)
react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 10")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

Hm... still tool calling but no response, maybe the assistant (llm) lacks a back connection from tools. Then tools always returns to the assistant (llm) but the assistant can decide to use the tool or to answer.

Now, we add one new step:

We connect the `Tools` node *back* to the `Assistant`, forming a loop.

* After the `assistant` node executes, `tools_condition` checks if the model's output is a tool call.
* If it is a tool call, the flow is directed to the `tools` node.
* The `tools` node connects back to `assistant`.
* This loop continues as long as the model decides to call tools.
* If the model response is not a tool call, the flow is directed to END, terminating the process.

In [ ]:
# this is what we already know
# Graph
builder = StateGraph(MessagesState)
# Define nodes: these do the work
builder.add_node("assistant", assistant)
# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
# add a tool node with our tools
builder.add_node("tools", ToolNode(tools))
# conditional edge
builder.add_conditional_edges(
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    tools_condition,
)


##### this is new #####
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 10")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

# Your Task

## Implement Admin Verification Tool and Agent

You are tasked with creating a system to verify if a user is an admin based on their provided username and passphrase. Your system should utilize a **tool** and an **agent** to process and respond to user requests.

#### Requirements:

1. **Verification Tool**:
   - Create a function named `verify_admin` that takes a username and passphrase as arguments and returns `True` if both are correct (`username="admin"` and `passphrase="adminpassword"`). Otherwise, return `False`.

2. **Agent**:
   - Implement an agent to interact with users. The agent must use the `verify_admin` tool to verify the admin credentials.
   - The agent should respond with an appropriate message based on the verification result:
     - If the credentials are correct, respond with: `"Verification successful! Welcome, admin."`
     - If the credentials are incorrect, respond with: `"Verification failed. Invalid credentials."`

3. **Messages Completion**:
   - Complete the final messages at the end of the code to ensure that the agent can respond to two user requests:
     - User 1: `"Please verify me: I am the admin and my passphrase is adminpassword"`
     - User 2: `"Please verify me: I am the admin and my passphrase is admin-assword"`


**Hint:** It is mostly copy and paste.

In [ ]:
LLM_NAME = "qwen3:8b"
BASE_URL = "http://127.0.0.1:11434/v1" # localhost
OPENAI_API_KEY = "ollama" #fake api key

llm = ChatOpenAI(
    model=LLM_NAME,
    base_url=BASE_URL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    seed=42
)

In [ ]:
@tool
def verify_admin(username: str, passphrase: str) -> bool:
    """Verifies if the username and passphrase are correct. Returns True if correct else False.

    Args:
        username: Username to verify
        passphrase: Passphrase to verify
    """

    if username == "admin" and passphrase == "adminpassword":
        return True
    return False

tools = [verify_admin, ]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing verifying the admin using his username and passphrase.")

In [ ]:
# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

In [ ]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)

builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# Show
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
messages = [HumanMessage(content="Please verify me: I am the admin and my passphrase is adminpassword")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()

In [ ]:
messages = [HumanMessage(content="Please verify me: I am the admin and my passphrase is admin-assword")]
messages = react_graph.invoke({"messages": messages})

for m in messages['messages']:
    m.pretty_print()